# Step 7 — Semantic Search

This notebook performs semantic search using embeddings created in Step 6.

Flow:

1. Load embedding model.
2. Create embedding for user query.
3. Fetch embeddings from SQLite.
4. Calculate cosine similarity.
5. Display Top-K most relevant chunks.

> At this stage we do not use `sqlite-vec`. Similarity calculations are done in Python for clarity.

In [8]:
from pathlib import Path
import sqlite3
import numpy as np
from sentence_transformers import SentenceTransformer

DB_PATH = Path("../data/linux_docs.db")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

MODEL_NAME = "BAAI/bge-small-en-v1.5"
model = SentenceTransformer(MODEL_NAME)

print("Model:", MODEL_NAME)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model: BAAI/bge-small-en-v1.5


## Enter query

In [9]:
query = "How do I partition the disk during Arch Linux installation?"
print(query)


How do I partition the disk during Arch Linux installation?


In [10]:
query_embedding = model.encode(
    query,
    normalize_embeddings=True
).astype(np.float32)

print("Embedding dimension:", query_embedding.shape[0])


Embedding dimension: 384


## Load embeddings from SQLite

In [11]:
cursor.execute('''
SELECT
    e.chunk_id,
    e.embedding,
    c.section,
    c.content
FROM embeddings e
JOIN chunks c
ON e.chunk_id = c.id
''')

rows = cursor.fetchall()
print("Embeddings:", len(rows))

Embeddings: 41


## Calculate cosine similarity

In [12]:
results = []

for row in rows:

    embedding = np.frombuffer(
        row["embedding"],
        dtype=np.float32
    )

    score = float(np.dot(query_embedding, embedding))

    results.append({
        "chunk_id": row["chunk_id"],
        "section": row["section"],
        "score": score,
        "content": row["content"]
    })

results = sorted(
    results,
    key=lambda x: x["score"],
    reverse=True
)


## Top 5 results

In [13]:
TOP_K = 5

for i, item in enumerate(results[:TOP_K], start=1):

    print("=" * 100)
    print(f"Rank    : {i}")
    print(f"Score   : {item['score']:.4f}")
    print(f"Section : {item['section']}")
    print()
    print(item["content"][:700])
    print()


Rank    : 1
Score   : 0.7738
Section : Introduction

This document is a guide for installing [Arch Linux](/title/Arch_Linux) using the live system booted from an installation medium made from an official installation image. Its accessibility features are described on the page [Install Arch Linux with accessibility options](/title/Install_Arch_Linux_with_accessibility_options). For alternative means of installation, see [Category:Installation process](/title/Category:Installation_process).

Before installing, it would be advised to view the [FAQ](/title/FAQ). For conventions used in this document, see [Help:Reading](/title/Help:Reading). In particular, code examples may contain placeholders (formatted in *italics*

This guide is kept concise an

Rank    : 2
Score   : 0.7717
Section : Boot loader

Choose a boot loader applicable to your partitioning scheme and install it. See the explanations and the comparison table in [Arch boot process#Boot loader](/title/Arch_boot_process#Boot_loader

## Summary

Our pipeline can now:

- Crawl pages
- Parse into Markdown
- Chunk text
- Generate embeddings
- Perform semantic search

The next step is to combine retrieval results into an LLM prompt
to build a simple RAG system.

In [14]:
conn.close()
print("Done ✅")

Done ✅
